In [ ]:
import os
import requests
from pathlib import Path

# -----------------------------------
# Configuration
# -----------------------------------

BASE_URL = "https://d37ci6vzurychx.cloudfront.net/trip-data"

DOWNLOAD_DIR = Path("/content/drive/MyDrive/NYC_Taxi_Project/raw_data")
DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)

TARGET_GB = 12

# -----------------------------------
# Current size already downloaded
# -----------------------------------

current_size = sum(
    f.stat().st_size
    for f in DOWNLOAD_DIR.glob("*.parquet")
) / (1024**3)

print(f"Current size: {current_size:.2f} GB")

# -----------------------------------
# Download files newest first
# -----------------------------------

for year in range(2025, 2013, -1):

    for month in range(12, 0, -1):

        filename = f"yellow_tripdata_{year}-{month:02d}.parquet"

        filepath = DOWNLOAD_DIR / filename

        if filepath.exists():
            print(f"Already exists: {filename}")
            continue

        url = f"{BASE_URL}/{filename}"

        try:

            # Get file size first
            head = requests.head(url, timeout=30)

            if head.status_code != 200:
                print(f"Not found: {filename}")
                continue

            file_size = int(
                head.headers.get("Content-Length", 0)
            ) / (1024**3)

            print(
                f"Downloading {filename} "
                f"({file_size:.2f} GB)"
            )

            response = requests.get(
                url,
                stream=True,
                timeout=120
            )

            with open(filepath, "wb") as f:
                for chunk in response.iter_content(
                    chunk_size=8 * 1024 * 1024
                ):
                    if chunk:
                        f.write(chunk)

            current_size += file_size

            print(
                f"Total size: "
                f"{current_size:.2f} GB"
            )

            if current_size >= TARGET_GB:
                print("\nTarget reached.")
                break

        except Exception as e:
            print(
                f"Failed {filename}: {e}"
            )

    if current_size >= TARGET_GB:
        break

print(
    f"\nFinal downloaded size: "
    f"{current_size:.2f} GB"
)


Current size: 0.00 GB
Total size: 0.07 GB
Total size: 0.13 GB
Total size: 0.20 GB
Total size: 0.27 GB
Total size: 0.33 GB
Total size: 0.39 GB
Total size: 0.46 GB
Total size: 0.53 GB
Total size: 0.60 GB
Total size: 0.66 GB
Total size: 0.72 GB
Total size: 0.77 GB
Total size: 0.83 GB
Total size: 0.89 GB
Total size: 0.95 GB
Total size: 1.00 GB
Total size: 1.05 GB
Total size: 1.10 GB
Total size: 1.16 GB
Total size: 1.21 GB
Total size: 1.27 GB
Total size: 1.32 GB
Total size: 1.37 GB
Total size: 1.42 GB
Total size: 1.47 GB
Total size: 1.52 GB
Total size: 1.58 GB
Total size: 1.62 GB
Total size: 1.67 GB
Total size: 1.71 GB
Total size: 1.76 GB
Total size: 1.82 GB
Total size: 1.87 GB
Total size: 1.92 GB
Total size: 1.97 GB
Total size: 2.01 GB
Total size: 2.06 GB
Total size: 2.11 GB
Total size: 2.16 GB
Total size: 2.21 GB
Total size: 2.25 GB
Total size: 2.30 GB
Total size: 2.35 GB
Total size: 2.40 GB
Total size: 2.45 GB
Total size: 2.51 GB
Total size: 2.55 GB
Total size: 2.58 GB
Total size: 2.63 G

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.types import *
from functools import reduce
import os

# Initialize SparkSession
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder.appName("NYC Taxi Data Cleaning")
    .config("spark.driver.memory", "15g")
    .config("spark.executor.memory", "15g")
    .getOrCreate()
)

RAW_DIR = "/content/drive/MyDrive/NYC_Taxi_Project/raw_data"

MONTHLY_OUTPUT = \
"/content/drive/MyDrive/NYC_Taxi_Project/cleaned_monthly"

COMBINED_OUTPUT = \
"/content/drive/MyDrive/NYC_Taxi_Project/cleaned_combined"

os.makedirs(MONTHLY_OUTPUT, exist_ok=True)

required_cols = {
    "vendorid": IntegerType(),
    "tpep_pickup_datetime": TimestampType(),
    "tpep_dropoff_datetime": TimestampType(),
    "passenger_count": IntegerType(),
    "trip_distance": DoubleType(),
    "pulocationid": IntegerType(),
    "dolocationid": IntegerType(),
    "fare_amount": DoubleType(),
    "tip_amount": DoubleType(),
    "total_amount": DoubleType()
}

cleaned_dfs = []

files = sorted(
    [
        f for f in os.listdir(RAW_DIR)
        if f.endswith(".parquet")
    ]
)

print(f"Files found: {len(files)}")

for file in files:

    print(f"\nProcessing {file}")

    df = (
        spark.read
        .option("mergeSchema", "true")
        .parquet(
            os.path.join(RAW_DIR, file)
        )
    )

    df = df.toDF(
        *[c.lower() for c in df.columns]
    )

    # Add missing columns

    for col_name, dtype in required_cols.items():

        if col_name not in df.columns:

            df = df.withColumn(
                col_name,
                F.lit(None).cast(dtype)
            )

    # Standardize datatypes

    for col_name, dtype in required_cols.items():

        df = df.withColumn(
            col_name,
            F.col(col_name).cast(dtype)
        )

    # Remove duplicates

    df = df.dropDuplicates([
        "vendorid",
        "tpep_pickup_datetime",
        "tpep_dropoff_datetime",
        "pulocationid",
        "dolocationid",
        "trip_distance",
        "fare_amount"
    ])

    # Remove critical nulls

    df = df.dropna(
        subset=[
            "tpep_pickup_datetime",
            "tpep_dropoff_datetime",
            "trip_distance"
        ]
    )

    # Distance filtering

    df = df.filter(
        (F.col("trip_distance") > 0) &
        (F.col("trip_distance") <= 100)
    )

    # Fare filtering

    df = df.filter(
        (F.col("fare_amount") > 0) &
        (F.col("fare_amount") <= 500)
    )

    # Passenger filtering

    if "passenger_count" in df.columns:

        df = df.filter(
            (F.col("passenger_count") > 0) &
            (F.col("passenger_count") <= 8)
        )

    # Location filtering

    df = df.filter(
        (F.col("pulocationid") > 0) &
        (F.col("dolocationid") > 0)
    )

    # Trip duration

    df = df.withColumn(
        "trip_duration_min",
        (
            F.unix_timestamp(
                "tpep_dropoff_datetime"
            )
            -
            F.unix_timestamp(
                "tpep_pickup_datetime"
            )
        ) / 60
    )

    df = df.filter(
        (F.col("trip_duration_min") > 0) &
        (F.col("trip_duration_min") <= 1440)
    )

    # Date variables

    df = (
        df
        .withColumn(
            "pickup_year",
            F.year(
                "tpep_pickup_datetime"
            )
        )
        .withColumn(
            "pickup_month",
            F.month(
                "tpep_pickup_datetime"
            )
        )
        .withColumn(
            "pickup_day",
            F.dayofmonth(
                "tpep_pickup_datetime"
            )
        )
        .withColumn(
            "pickup_hour",
            F.hour(
                "tpep_pickup_datetime"
            )
        )
        .withColumn(
            "pickup_dayofweek",
            F.dayofweek(
                "tpep_pickup_datetime"
            )
        )
    )

    monthly_output = os.path.join(
        MONTHLY_OUTPUT,
        file.replace(
            ".parquet",
            "_cleaned"
        )
    )

    (
        df.write
        .mode("overwrite")
        .parquet(monthly_output)
    )

    print(f"Saved: {monthly_output}")

    cleaned_dfs.append(df)

Files found: 139

Processing yellow_tripdata_2014-06.parquet
Saved: /content/drive/MyDrive/NYC_Taxi_Project/cleaned_monthly/yellow_tripdata_2014-06_cleaned

Processing yellow_tripdata_2014-07.parquet
Saved: /content/drive/MyDrive/NYC_Taxi_Project/cleaned_monthly/yellow_tripdata_2014-07_cleaned

Processing yellow_tripdata_2014-08.parquet
Saved: /content/drive/MyDrive/NYC_Taxi_Project/cleaned_monthly/yellow_tripdata_2014-08_cleaned

Processing yellow_tripdata_2014-09.parquet
Saved: /content/drive/MyDrive/NYC_Taxi_Project/cleaned_monthly/yellow_tripdata_2014-09_cleaned

Processing yellow_tripdata_2014-10.parquet
Saved: /content/drive/MyDrive/NYC_Taxi_Project/cleaned_monthly/yellow_tripdata_2014-10_cleaned

Processing yellow_tripdata_2014-11.parquet
Saved: /content/drive/MyDrive/NYC_Taxi_Project/cleaned_monthly/yellow_tripdata_2014-11_cleaned

Processing yellow_tripdata_2014-12.parquet
Saved: /content/drive/MyDrive/NYC_Taxi_Project/cleaned_monthly/yellow_tripdata_2014-12_cleaned

Processin